In [4]:
# =========================================================
# MODEL EVALUATION
# =========================================================

import sys
sys.path.append("../src")

import pandas as pd

country = "Norway"
forecast_year = 2025

comparison_plot_df = pd.read_csv(
    f"../outputs/tables/{country}_comparison_plot_df_{forecast_year}.csv",
    index_col=0,
    parse_dates=True
)

comparison_plot_df.head()

from evaluation_utils import (
    calculate_annual_metrics,
    calculate_monthly_metrics,
    run_volatility_analysis,
    run_dm_tests,
    save_evaluation_outputs
)

In [7]:
df = pd.read_csv("../data/processed_data.csv", index_col=0, parse_dates=True)
df = df.sort_index()
df = df.asfreq("D")
df = df.dropna()

In [5]:
# =========================================================
# 1. ANNUAL METRICS
# =========================================================

annual_table = calculate_annual_metrics(
    comparison_plot_df=comparison_plot_df,
    country=country,
    forecast_year=forecast_year
)

annual_table

,Country,Forecast Year,Model,RMSE (log),MAE (log),Rank
0,Norway,2025,SARIMAX Restricted,0.022200,0.017373,1
1,Norway,2025,SARIMAX Full,0.022728,0.018065,2
2,Norway,2025,XGBoost No Weather,0.024621,0.019349,3
3,Norway,2025,XGBoost Full,0.025818,0.020802,4
4,Norway,2025,XGBoost Restricted,0.026010,0.020992,5
5,Norway,2025,Seasonal Naïve,0.066772,0.049301,6


In [14]:
# =========================================================
# 2. MONTHLY METRICS
# =========================================================

monthly_table = calculate_monthly_metrics(
    comparison_plot_df=comparison_plot_df,
    country=country,
    forecast_year=forecast_year
)

monthly_table.head()

output_path = f"../outputs/tables/{country}_monthly_metrics_{forecast_year}.csv"

monthly_table.to_csv(output_path, index=False)

print(f"Saved: {output_path}")

Saved: ../outputs/tables/Norway_monthly_metrics_2025.csv


In [8]:
# =========================================================
# 3. VOLATILITY ANALYSIS
# =========================================================

summary_counts_all_df, performance_by_type_all_df, winners_all_df, plot_ready_tables = run_volatility_analysis(
    comparison_plot_df=comparison_plot_df,
    df=df,
    country=country,
    forecast_year=forecast_year,
    volatility_percentile_list=[90, 80, 75],
    temp_percentile_list=[90, 80, 75],
    pair_scenarios=True
)

performance_by_type_all_df.head()

,Country,Forecast Year,Volatility Percentile,Temperature Percentile,Volatility Type,Days,Model,RMSE,MAE
0,Norway,2025,90,90,Other Factors,30,XGBoost Full,0.021,0.017
1,Norway,2025,90,90,Other Factors,30,XGBoost Restricted,0.021,0.016
2,Norway,2025,90,90,Other Factors,30,XGBoost No Weather,0.024,0.019
3,Norway,2025,90,90,Other Factors,30,SARIMAX Full,0.030,0.023
4,Norway,2025,90,90,Other Factors,30,SARIMAX Restricted,0.032,0.024


In [9]:
winners_all_df

,Volatility Type,Model,Days,RMSE,MAE,Country,Forecast Year,Volatility Percentile,Temperature Percentile
0,Other Factors,XGBoost Full,30,0.021,0.017,Norway,2025,90,90
1,Temperature-Driven,SARIMAX Full,7,0.012,0.008,Norway,2025,90,90
2,Other Factors,XGBoost Full,55,0.023,0.018,Norway,2025,80,80
3,Temperature-Driven,SARIMAX Full,18,0.018,0.014,Norway,2025,80,80
4,Other Factors,XGBoost Full,62,0.023,0.018,Norway,2025,75,75
5,Temperature-Driven,SARIMAX Full,29,0.018,0.014,Norway,2025,75,75


In [10]:
# =========================================================
# 4. DIEBOLD-MARIANO TESTS
# =========================================================

dm_table_clean = run_dm_tests(
    comparison_plot_df=comparison_plot_df,
    country=country,
    forecast_year=forecast_year,
    winter_months=[12, 1, 2, 3],
    lag=None,
    h=1,
    power=2
)

dm_table_clean

,Country,Forecast Year,Period,Model 1,Model 2,DM Stat,p-value (stars),Winner (5%)
0,Norway,2025,Full Year,SARIMAX Restricted,XGBoost Restricted,-1.894,0.0590*,No significant difference
1,Norway,2025,Full Year,SARIMAX Restricted,XGBoost Full,-1.863,0.0633*,No significant difference
2,Norway,2025,Full Year,SARIMAX Full,XGBoost Restricted,-1.679,0.0941*,No significant difference
3,Norway,2025,Full Year,SARIMAX Full,XGBoost Full,-1.639,0.1021,No significant difference
4,Norway,2025,Full Year,SARIMAX Restricted,XGBoost No Weather,-1.383,0.1674,No significant difference
5,Norway,2025,Full Year,SARIMAX Full,SARIMAX Restricted,1.157,0.2481,No significant difference
6,Norway,2025,Full Year,SARIMAX Full,XGBoost No Weather,-1.099,0.2725,No significant difference
7,Norway,2025,Full Year,XGBoost Restricted,XGBoost No Weather,0.913,0.3621,No significant difference
8,Norway,2025,Full Year,XGBoost Full,XGBoost No Weather,0.824,0.4106,No significant difference
9,Norway,2025,Full Year,XGBoost Full,XGBoost Restricted,-0.730,0.4661,No significant difference


In [13]:
# =========================================================
# 5. SAVE TABLES
# =========================================================

excel_path = save_evaluation_outputs(
    country=country,
    forecast_year=forecast_year,
    annual_table=annual_table,
    monthly_table=monthly_table,
    summary_counts_all_df=summary_counts_all_df,
    performance_by_type_all_df=performance_by_type_all_df,
    winners_all_df=winners_all_df,
    dm_table_clean=dm_table_clean,
    output_dir="../outputs/tables"
)

print(f"Saved: {excel_path}")

Saved: ../outputs/tables\Norway_model_evaluation_2025.xlsx
